# verify — LLaVA 13B (Local Ollama)

Sends questions from `questions.json` (the main QuestionBank dataset) to **llava:13b** running via Ollama inside Colab.

**Data setup:** Mount your Google Drive and point `DRIVE_ROOT` to the `QuestionBank/` folder, which must contain:
```
Data/
  questions.json
  questions/   ← .txt files with question text
  images/      ← image files referenced in questions.json
```

Answers are written to `/content/outputs/<id>.txt`.

> **Runtime:** GPU (T4 or better) recommended.

In [ ]:
!pip install -q ollama

In [ ]:
import subprocess, time, os

!curl -fsSL https://ollama.com/install.sh | sh

proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print("Ollama service started.")

!ollama pull llava:13b
print("Model ready.")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# ── Set this to where QuestionBank/ lives on your Drive ──────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/QuestionBank"
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR  = os.path.join(DRIVE_ROOT, "Data")
JSON_FILE = os.path.join(DATA_DIR, "questions.json")

assert os.path.isfile(JSON_FILE), f"Not found: {JSON_FILE}"
print("Found questions.json ✓")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL           = "llava:13b"
OUTPUT_DIR      = "/content/outputs/verify"
POST_CALL_DELAY = 3

SYSTEM_PROMPT = (
    "You are expert computer science tutor. "
    "Answer the given question step by step. "
    "Begin by explaining your reasoning process clearly. "
    "Think step by step before answering the question."
)

FILTER_IDS = None   # e.g. ["1", "5", "12"]
LAST_N     = None   # e.g. 10

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json, time
from ollama import Client

client = Client(host="http://localhost:11434")

def load_entries():
    with open(JSON_FILE, encoding="utf-8") as f:
        return json.load(f)

def read_question_file(rel_path):
    full = os.path.join(DATA_DIR, rel_path.lstrip("./"))
    if not os.path.exists(full):
        return ""
    with open(full, encoding="utf-8") as f:
        return f.read().strip()

def ask(entry):
    out_path = os.path.join(OUTPUT_DIR, f"{entry['id']}.txt")
    if os.path.exists(out_path):
        print(f"  [SKIP] #{entry['id']} — already answered")
        return

    question_text = read_question_file(entry["question"])
    if not question_text:
        print(f"  [SKIP] #{entry['id']} — question file missing")
        return

    image_paths = []
    for rel in entry.get("images", []):
        full = os.path.join(DATA_DIR, rel.lstrip("./"))
        if os.path.exists(full):
            image_paths.append(full)

    user_msg = {"role": "user", "content": question_text}
    if image_paths:
        user_msg["images"] = image_paths

    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            user_msg,
        ],
        options={"temperature": 0},
    )
    answer = response["message"]["content"].strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(answer)
    print(f"  [OK]   #{entry['id']} — {entry.get('topic', '')} → {out_path}")

print("Helpers defined.")

In [ ]:
all_entries = load_entries()

if FILTER_IDS:
    id_set  = set(str(i) for i in FILTER_IDS)
    entries = [e for e in all_entries if str(e["id"]) in id_set]
elif LAST_N:
    entries = all_entries[-LAST_N:]
else:
    entries = all_entries

print(f"Model  : {MODEL}")
print(f"Output : {OUTPUT_DIR}")
print(f"Running {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}…")

In [ ]:
for i, entry in enumerate(entries):
    try:
        ask(entry)
    except Exception as exc:
        print(f"  [ERR]  #{entry['id']} — {exc}")
    if i < len(entries) - 1:
        time.sleep(POST_CALL_DELAY)

print("\nDone.")

In [ ]:
# Copy outputs back to Drive (optional)
import shutil
dest = os.path.join(DRIVE_ROOT, "verify", "llava:13b")
os.makedirs(dest, exist_ok=True)
for f in os.listdir(OUTPUT_DIR):
    shutil.copy(os.path.join(OUTPUT_DIR, f), os.path.join(dest, f))
print(f"Outputs copied to Drive: {dest}")